# Python Variables, Objects and References — Practical Experiments

Companion notebook to *"Python Variables, Objects and References: The Mental Model You Actually Need."*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imblessed-tech/articles-publications/blob/main/notebook/01_Python_variables_objects_and_references.ipynb)


### 1. Names & Binding Basics

Assignment binds a name to an object. It does not "store a value in a box."


In [1]:
x = 10
print(type(x))
print(id(x))

# Rebind x to a different object
x = "now a string"
print(type(x))
print(id(x))


<class 'int'>
140724745172168
<class 'str'>
2848681896752


Notice: `id(x)` changes after rebinding — `x` is now pointing at a completely different object, not the same box with new contents.

### 2. Aliasing

Two names can refer to the **same** object.


In [2]:
a = [1, 2, 3]
b = a

b.append(4)

print("a:", a)
print("b:", b)
print("a is b:", a is b)


a: [1, 2, 3, 4]
b: [1, 2, 3, 4]
a is b: True


### 3. Mutation vs. Rebinding

Same starting object. One function mutates it. One function rebinds its local name. Only one is visible to the caller.


In [3]:
def mutate(lst):
    lst.append(99)          # mutates the object

def rebind(lst):
    lst = lst + [99]        # rebinds the LOCAL name only

data = [1, 2, 3]
mutate(data)
print("after mutate:", data)   # caller sees the change

data = [1, 2, 3]
rebind(data)
print("after rebind:", data)   # caller sees NOTHING


after mutate: [1, 2, 3, 99]
after rebind: [1, 2, 3]


### 4. Mutable vs. Immutable Objects

`+=` behaves differently depending on whether the object is mutable.


In [4]:
# Mutable: list
a = [1, 2]
b = a
print("before:", id(a))
a += [3]                # in-place mutation (__iadd__)
print("after: ", id(a))
print("b also changed:", b)


before: 2848681690304
after:  2848681690304
b also changed: [1, 2, 3]


In [5]:
# Immutable: int
x = 10
y = x
print("before:", id(x))
x += 5                   # no __iadd__ on int -> falls back to __add__, rebinds
print("after: ", id(x))
print("y unaffected:", y)


before: 140724745172168
after:  140724745172328
y unaffected: 10


Same `+=` syntax. Opposite mechanism. The object's type — not the operator — decides whether this mutates or rebinds.

### 5. `is` vs. `==`

`==` asks "equal in value?" `is` asks "the same object?"


In [6]:
a = [1, 2]
b = a
c = [1, 2]

print("a is b:", a is b)   # True  - same object
print("a is c:", a is c)   # False - different object
print("a == c:", a == c)   # True  - equal value

print(id(a), id(b), id(c))


a is b: True
a is c: False
a == c: True
2848681704512 2848681704512 2848681913216


**Bonus gotcha — small integer / string interning:**

CPython caches small integers and some strings, so `is` can return `True` even without explicit aliasing. This is an implementation detail, not something to rely on.


In [7]:
p = 5
q = 5
print("p is q:", p is q)     # True (small int cache)

m = 1000
n = 1000
print("m is n:", m is n)     # Often False (outside the cache range)

# Never use `is` to compare values for this reason -- always use `==`


p is q: True
m is n: False


### 6. Copying

To get an independent object, you must ask for one explicitly.


In [8]:
a = [1, 2, 3]
b = a.copy()          # shallow copy -- new outer object
b.append(4)

print("a:", a)
print("b:", b)
print("a is b:", a is b)


a: [1, 2, 3]
b: [1, 2, 3, 4]
a is b: False


**The shallow copy gotcha:** a shallow copy only copies the *outer* container. Nested mutable objects are still shared.


In [9]:
a = [[1, 2], [3, 4]]
b = a.copy()

print("outer a is b:", a is b)          # False -- different outer list
print("inner a[0] is b[0]:", a[0] is b[0])  # True -- SAME inner list!

b[0].append(5)
print("a:", a)   # a changed too, even though we only touched b


outer a is b: False
inner a[0] is b[0]: True
a: [[1, 2, 5], [3, 4]]


In [10]:
import copy

a = [[1, 2], [3, 4]]
b = copy.deepcopy(a)

print("inner a[0] is b[0]:", a[0] is b[0])  # False -- independent now

b[0].append(5)
print("a:", a)   # untouched
print("b:", b)


inner a[0] is b[0]: False
a: [[1, 2], [3, 4]]
b: [[1, 2, 5], [3, 4]]


### 7. Function Arguments Are Object References, Not "By Value" or "By Reference"

Whether a function's effect is visible to the caller depends entirely on mutate-vs-rebind inside the function body -- not on any special calling convention.


In [11]:
def add_item(lst, item):
    lst.append(item)      # mutates -- caller sees it
    return lst

original = [1, 2, 3]
result = add_item(original, 4)

print("original:", original)
print("same object:", original is result)


original: [1, 2, 3, 4]
same object: True


### 8. The Mutable Default Argument Trap

Default values are evaluated **once**, at `def` time, and live on the function object.


In [12]:
def add_to_bucket(item, bucket=[]):
    bucket.append(item)
    return bucket

print(add_to_bucket("a"))   # ['a']
print(add_to_bucket("b"))   # ['a', 'b']  <- surprise: same list every call!

print(add_to_bucket.__defaults__)


['a']
['a', 'b']
(['a', 'b'],)


In [13]:
# The fix
def add_to_bucket_fixed(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(add_to_bucket_fixed("a"))   # ['a']
print(add_to_bucket_fixed("b"))   # ['b']  <- fresh list each call


['a']
['b']
